# Oil & Gas Production Optimization - Synthetic Data Generator

## Overview
This notebook generates synthetic data for oil & gas production monitoring:
- **Well production telemetry** - Pressure, temperature, flow rates
- **Artificial lift data** - ESP and rod pump performance
- **Drilling parameters** - ROP, WOB, torque, mud properties
- **Production events** - Shutdowns, workovers, tests

## Tables Created
| Table | Description | Volume |
|-------|-------------|--------|
| `dim_wells` | Well registry with completions data | ~200 rows |
| `dim_equipment` | Artificial lift equipment | ~300 rows |
| `fact_production_telemetry` | Wellhead sensor readings | 500K+ rows/day |
| `fact_esp_telemetry` | ESP performance data | 200K+ rows/day |
| `fact_rod_pump_cards` | Dynamometer cards | ~50K rows/day |
| `fact_drilling_parameters` | Real-time drilling data | 100K+ rows/day |
| `fact_production_events` | Operational events | ~1K rows/month |

## How to Run in Fabric
1. Attach this notebook to a Lakehouse
2. Run all cells sequentially
3. Data will be written as Delta tables

In [ ]:
!pip install faker

In [ ]:
# Configuration
SEED = 42
NUM_FIELDS = 5
WELLS_PER_FIELD = 40

from datetime import datetime, timedelta
START_DATE = datetime(2024, 9, 1)
END_DATE = datetime(2024, 9, 8)

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import uuid

np.random.seed(SEED)
random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

print("Libraries loaded successfully")

## 1. Generate Well Dimension

In [ ]:
def generate_wells(num_fields, wells_per_field):
    wells = []
    
    basins = ['Permian', 'Eagle Ford', 'Bakken', 'DJ Basin', 'Marcellus']
    lift_types = ['ESP', 'Rod Pump', 'Gas Lift', 'Plunger Lift', 'Flowing']
    
    for f in range(num_fields):
        field_name = f'{fake.last_name()} Field'
        basin = basins[f % len(basins)]
        
        for w in range(wells_per_field):
            well_type = random.choices(['Oil', 'Gas', 'Oil & Gas'], weights=[0.5, 0.3, 0.2])[0]
            lift_type = random.choices(lift_types, weights=[0.35, 0.30, 0.20, 0.10, 0.05])[0]
            
            # Realistic production rates based on lift type and age
            spud_date = fake.date_between(start_date='-10y', end_date='-6m')
            age_years = (datetime.now() - pd.to_datetime(spud_date)).days / 365
            
            # Initial rate and decline
            initial_oil = random.uniform(200, 2000) if well_type in ['Oil', 'Oil & Gas'] else 0
            initial_gas = random.uniform(500, 5000) if well_type in ['Gas', 'Oil & Gas'] else random.uniform(100, 500)
            decline_rate = random.uniform(0.15, 0.35)  # Annual decline
            
            current_oil = initial_oil * (1 - decline_rate) ** age_years
            current_gas = initial_gas * (1 - decline_rate * 0.8) ** age_years
            
            wells.append({
                'well_id': f'WELL-{str(f+1).zfill(2)}-{str(w+1).zfill(3)}',
                'well_name': f'{field_name} #{w+1}',
                'field_name': field_name,
                'basin': basin,
                'api_number': f'{random.randint(10, 50)}-{random.randint(100, 999)}-{random.randint(10000, 99999)}',
                'latitude': round(random.uniform(28, 48), 6),
                'longitude': round(random.uniform(-105, -95), 6),
                'well_type': well_type,
                'lift_type': lift_type,
                'spud_date': spud_date,
                'first_production_date': spud_date + timedelta(days=random.randint(30, 180)),
                'total_depth_ft': random.randint(8000, 22000),
                'lateral_length_ft': random.randint(5000, 15000) if random.random() > 0.2 else 0,
                'initial_oil_bopd': round(initial_oil, 0),
                'initial_gas_mcfd': round(initial_gas, 0),
                'current_oil_bopd': round(current_oil, 0),
                'current_gas_mcfd': round(current_gas, 0),
                'water_cut_pct': round(random.uniform(20, 85), 1),
                'reservoir_pressure_psi': random.randint(1500, 6000),
                'bottomhole_temp_f': random.randint(150, 300),
                'status': random.choices(['Producing', 'Shut-in', 'Workover', 'P&A'], weights=[0.85, 0.08, 0.05, 0.02])[0]
            })
    
    return pd.DataFrame(wells)

df_wells = generate_wells(NUM_FIELDS, WELLS_PER_FIELD)
print(f"Generated {len(df_wells)} wells")
print(f"By lift type: {df_wells['lift_type'].value_counts().to_dict()}")
df_wells.head()

In [ ]:
def generate_equipment(wells_df):
    equipment = []
    
    # Filter to wells with artificial lift
    lift_wells = wells_df[wells_df['lift_type'].isin(['ESP', 'Rod Pump', 'Gas Lift'])]
    
    for _, well in lift_wells.iterrows():
        install_date = fake.date_between(start_date=well['first_production_date'], end_date='today')
        
        if well['lift_type'] == 'ESP':
            equipment.append({
                'equipment_id': f'{well["well_id"]}-ESP-001',
                'well_id': well['well_id'],
                'equipment_type': 'ESP',
                'manufacturer': random.choice(['Schlumberger', 'Baker Hughes', 'Halliburton', 'Weatherford']),
                'model': f'ESP-{random.randint(400, 1000)}',
                'install_date': install_date,
                'design_rate_bfpd': random.randint(500, 5000),
                'design_head_ft': random.randint(3000, 12000),
                'motor_hp': random.choice([50, 75, 100, 150, 200, 300]),
                'stages': random.randint(80, 300),
                'pump_depth_ft': well['total_depth_ft'] - random.randint(500, 2000),
                'run_life_days': (datetime.now() - pd.to_datetime(install_date)).days,
                'status': random.choices(['Running', 'Failed', 'Pulled'], weights=[0.85, 0.10, 0.05])[0]
            })
        elif well['lift_type'] == 'Rod Pump':
            equipment.append({
                'equipment_id': f'{well["well_id"]}-RP-001',
                'well_id': well['well_id'],
                'equipment_type': 'Rod Pump',
                'manufacturer': random.choice(['Lufkin', 'Weatherford', 'Dover', 'National Oilwell']),
                'model': f'RP-{random.randint(100, 500)}',
                'install_date': install_date,
                'design_rate_bfpd': random.randint(50, 500),
                'stroke_length_in': random.choice([54, 64, 74, 86, 100, 120]),
                'strokes_per_min': round(random.uniform(4, 12), 1),
                'pump_diameter_in': random.choice([1.5, 1.75, 2.0, 2.25, 2.5]),
                'pump_depth_ft': well['total_depth_ft'] - random.randint(500, 3000),
                'run_life_days': (datetime.now() - pd.to_datetime(install_date)).days,
                'status': random.choices(['Running', 'Failed', 'Pulled'], weights=[0.88, 0.08, 0.04])[0]
            })
        elif well['lift_type'] == 'Gas Lift':
            equipment.append({
                'equipment_id': f'{well["well_id"]}-GL-001',
                'well_id': well['well_id'],
                'equipment_type': 'Gas Lift',
                'manufacturer': random.choice(['Weatherford', 'Schlumberger', 'Baker Hughes']),
                'model': f'GL-{random.randint(100, 300)}',
                'install_date': install_date,
                'design_rate_bfpd': random.randint(200, 3000),
                'num_valves': random.randint(3, 8),
                'injection_depth_ft': well['total_depth_ft'] - random.randint(1000, 4000),
                'injection_rate_mcfd': random.randint(100, 1000),
                'operating_pressure_psi': random.randint(800, 2500),
                'run_life_days': (datetime.now() - pd.to_datetime(install_date)).days,
                'status': random.choices(['Running', 'Off', 'Maintenance'], weights=[0.90, 0.05, 0.05])[0]
            })
    
    return pd.DataFrame(equipment)

df_equipment = generate_equipment(df_wells)
print(f"Generated {len(df_equipment)} equipment records")
print(f"By type: {df_equipment['equipment_type'].value_counts().to_dict()}")
df_equipment.head()

## 2. Generate Production Telemetry

In [ ]:
def generate_production_telemetry(wells_df, start_time, hours=24):
    """Generate wellhead sensor readings"""
    telemetry = []
    
    producing_wells = wells_df[wells_df['status'] == 'Producing'].head(50)  # Limit for demo
    
    for _, well in producing_wells.iterrows():
        current_time = start_time
        
        # Base values
        base_tubing_pressure = random.uniform(100, 800)
        base_casing_pressure = random.uniform(50, 400)
        base_line_pressure = random.uniform(50, 200)
        
        while current_time < start_time + timedelta(hours=hours):
            # Add realistic variation
            tubing_pressure = base_tubing_pressure + np.random.normal(0, 10)
            casing_pressure = base_casing_pressure + np.random.normal(0, 5)
            line_pressure = base_line_pressure + np.random.normal(0, 3)
            
            # Flow rates with some variation
            oil_rate = well['current_oil_bopd'] * random.uniform(0.9, 1.1) / 24  # Hourly
            gas_rate = well['current_gas_mcfd'] * random.uniform(0.9, 1.1) / 24
            water_rate = oil_rate * (well['water_cut_pct'] / (100 - well['water_cut_pct']))
            
            # Temperature
            wellhead_temp = random.uniform(80, 150)
            
            # Random anomalies (2% probability)
            has_anomaly = random.random() < 0.02
            if has_anomaly:
                anomaly_type = random.choice(['pressure_spike', 'rate_drop', 'temp_high'])
                if anomaly_type == 'pressure_spike':
                    tubing_pressure *= random.uniform(1.3, 1.8)
                elif anomaly_type == 'rate_drop':
                    oil_rate *= random.uniform(0.3, 0.6)
                    gas_rate *= random.uniform(0.3, 0.6)
                else:
                    wellhead_temp *= random.uniform(1.2, 1.5)
            
            telemetry.append({
                'reading_id': str(uuid.uuid4()),
                'well_id': well['well_id'],
                'timestamp': current_time,
                'tubing_pressure_psi': round(tubing_pressure, 1),
                'casing_pressure_psi': round(casing_pressure, 1),
                'line_pressure_psi': round(line_pressure, 1),
                'wellhead_temp_f': round(wellhead_temp, 1),
                'choke_size_64ths': random.choice([16, 24, 32, 48, 64]),
                'oil_rate_bph': round(oil_rate, 2),
                'gas_rate_mcfh': round(gas_rate, 2),
                'water_rate_bph': round(water_rate, 2),
                'has_anomaly': has_anomaly
            })
            
            current_time += timedelta(minutes=5)
    
    return pd.DataFrame(telemetry)

df_production = generate_production_telemetry(df_wells, START_DATE, hours=24)
print(f"Generated {len(df_production)} production telemetry records")
print(f"Anomalies: {df_production['has_anomaly'].sum()}")
df_production.head()

## 3. Generate ESP Telemetry

In [ ]:
def generate_esp_telemetry(equipment_df, wells_df, start_time, hours=24):
    """Generate ESP performance data"""
    telemetry = []
    
    esp_equipment = equipment_df[equipment_df['equipment_type'] == 'ESP']
    
    for _, esp in esp_equipment.iterrows():
        well = wells_df[wells_df['well_id'] == esp['well_id']].iloc[0]
        
        # Calculate degradation based on run life
        run_life = esp['run_life_days']
        degradation = min(1.0, run_life / 1500)  # Full degradation at 1500 days
        
        current_time = start_time
        while current_time < start_time + timedelta(hours=hours):
            # Motor parameters
            base_current = esp['motor_hp'] * 0.8  # Rough amps estimate
            motor_current = base_current * (1 + degradation * 0.2) + np.random.normal(0, 2)
            motor_voltage = 480 + np.random.normal(0, 10)
            motor_temp = 180 + degradation * 30 + np.random.normal(0, 5)
            
            # Pump parameters
            intake_pressure = random.uniform(100, 400) + np.random.normal(0, 10)
            discharge_pressure = intake_pressure + esp['design_head_ft'] * 0.43 * random.uniform(0.8, 1.0)
            
            # Frequency (VFD controlled)
            frequency = random.choice([50, 55, 60, 65]) + np.random.normal(0, 1)
            
            # Flow rate
            efficiency = 1 - degradation * 0.3
            flow_rate = esp['design_rate_bfpd'] * efficiency * random.uniform(0.7, 1.0)
            
            # Vibration increases with degradation
            vibration = 0.1 + degradation * 0.3 + np.random.exponential(0.05)
            
            # Failure prediction flag
            failure_risk = degradation > 0.8 or motor_temp > 250 or vibration > 0.4
            
            telemetry.append({
                'reading_id': str(uuid.uuid4()),
                'equipment_id': esp['equipment_id'],
                'well_id': esp['well_id'],
                'timestamp': current_time,
                'motor_current_a': round(motor_current, 1),
                'motor_voltage_v': round(motor_voltage, 1),
                'motor_temp_f': round(motor_temp, 1),
                'intake_pressure_psi': round(intake_pressure, 1),
                'discharge_pressure_psi': round(discharge_pressure, 1),
                'frequency_hz': round(frequency, 1),
                'flow_rate_bfpd': round(flow_rate, 0),
                'vibration_g': round(vibration, 3),
                'power_kw': round(motor_current * motor_voltage * 1.732 / 1000, 1),
                'efficiency_pct': round(efficiency * 100, 1),
                'failure_risk_flag': failure_risk
            })
            
            current_time += timedelta(minutes=1)
    
    return pd.DataFrame(telemetry)

df_esp = generate_esp_telemetry(df_equipment, df_wells, START_DATE, hours=6)  # 6 hours for demo
print(f"Generated {len(df_esp)} ESP telemetry records")
print(f"Failure risks: {df_esp['failure_risk_flag'].sum()}")
df_esp.head()

## 4. Generate Rod Pump Dynamometer Cards

In [ ]:
def generate_pump_card(card_type='normal'):
    """Generate a simplified dynamometer card shape"""
    positions = np.linspace(0, 100, 50)  # 50 points per stroke
    
    if card_type == 'normal':
        # Rectangular card (healthy pump)
        loads = 3000 + 1500 * np.sin(positions * np.pi / 100) + np.random.normal(0, 100, 50)
    elif card_type == 'gas_interference':
        # Compressed card
        loads = 2500 + 500 * np.sin(positions * np.pi / 100) + np.random.normal(0, 150, 50)
    elif card_type == 'fluid_pound':
        # Sharp drop mid-stroke
        loads = 3000 + 1500 * np.sin(positions * np.pi / 100)
        loads[25:35] -= 1000  # Sharp drop
        loads += np.random.normal(0, 100, 50)
    elif card_type == 'worn_pump':
        # Slippage pattern
        loads = 2800 + 1000 * np.sin(positions * np.pi / 100) + np.random.normal(0, 200, 50)
    else:
        loads = 3000 + np.random.normal(0, 500, 50)
    
    return positions.tolist(), loads.tolist()

def generate_rod_pump_cards(equipment_df, start_time, hours=24):
    """Generate rod pump dynamometer card data"""
    cards = []
    
    rod_pumps = equipment_df[equipment_df['equipment_type'] == 'Rod Pump']
    
    for _, pump in rod_pumps.iterrows():
        current_time = start_time
        
        # Assign a condition to each pump
        pump_condition = random.choices(
            ['normal', 'gas_interference', 'fluid_pound', 'worn_pump'],
            weights=[0.7, 0.12, 0.10, 0.08]
        )[0]
        
        while current_time < start_time + timedelta(hours=hours):
            positions, loads = generate_pump_card(pump_condition)
            
            # Calculate card metrics
            peak_load = max(loads)
            min_load = min(loads)
            avg_load = np.mean(loads)
            card_area = np.trapz(loads, positions)
            
            # Fillage estimate
            fillage = 100 if pump_condition == 'normal' else random.uniform(50, 90)
            
            cards.append({
                'card_id': str(uuid.uuid4()),
                'equipment_id': pump['equipment_id'],
                'well_id': pump['well_id'],
                'timestamp': current_time,
                'card_type': 'Surface',
                'position_data': positions,
                'load_data': loads,
                'peak_load_lbs': round(peak_load, 0),
                'min_load_lbs': round(min_load, 0),
                'avg_load_lbs': round(avg_load, 0),
                'card_area': round(card_area, 0),
                'strokes_per_min': pump['strokes_per_min'],
                'stroke_length_in': pump['stroke_length_in'],
                'fillage_pct': round(fillage, 1),
                'diagnosed_condition': pump_condition,
                'requires_action': pump_condition != 'normal'
            })
            
            current_time += timedelta(hours=1)  # One card per hour
    
    return pd.DataFrame(cards)

df_cards = generate_rod_pump_cards(df_equipment, START_DATE, hours=24)
print(f"Generated {len(df_cards)} dynamometer cards")
print(f"Conditions: {df_cards['diagnosed_condition'].value_counts().to_dict()}")
df_cards.head()

## 5. Generate Drilling Parameters

In [ ]:
def generate_drilling_parameters(num_wells=5, start_time=None, hours=24):
    """Generate real-time drilling data for active wells"""
    drilling = []
    
    for w in range(num_wells):
        well_id = f'DRILL-{str(w+1).zfill(3)}'
        target_depth = random.randint(15000, 25000)
        current_depth = random.randint(5000, 12000)  # Somewhere in progress
        
        current_time = start_time
        while current_time < start_time + timedelta(hours=hours):
            # Rate of penetration varies by formation
            rop = random.uniform(30, 150)  # ft/hr
            
            # Drilling parameters
            wob = random.uniform(10, 35)  # klbs
            rpm = random.uniform(80, 180)
            torque = random.uniform(5, 25)  # kft-lbs
            
            # Mud parameters
            mud_weight = random.uniform(9, 14)  # ppg
            flow_rate = random.uniform(300, 600)  # gpm
            standpipe_pressure = random.uniform(2000, 4000)  # psi
            
            # Gas detection
            total_gas = max(0, np.random.exponential(50))  # units
            
            # Kick indicator (very rare)
            pit_gain = 0
            kick_indicator = False
            if random.random() < 0.005:  # 0.5% chance
                pit_gain = random.uniform(5, 30)  # barrels
                kick_indicator = True
            
            drilling.append({
                'reading_id': str(uuid.uuid4()),
                'well_id': well_id,
                'timestamp': current_time,
                'bit_depth_ft': round(current_depth, 0),
                'hole_depth_ft': round(current_depth, 0),
                'rop_fph': round(rop, 1),
                'wob_klbs': round(wob, 1),
                'rpm': round(rpm, 0),
                'torque_kftlbs': round(torque, 1),
                'mud_weight_ppg': round(mud_weight, 2),
                'flow_rate_gpm': round(flow_rate, 0),
                'standpipe_pressure_psi': round(standpipe_pressure, 0),
                'total_gas_units': round(total_gas, 1),
                'pit_volume_bbl': round(random.uniform(400, 600), 0),
                'pit_gain_bbl': round(pit_gain, 1),
                'ecd_ppg': round(mud_weight + random.uniform(0.1, 0.5), 2),
                'kick_indicator': kick_indicator,
                'drilling_state': random.choices(['Drilling', 'Tripping', 'Circulating', 'Connection'],
                                                 weights=[0.7, 0.1, 0.1, 0.1])[0]
            })
            
            # Progress depth
            current_depth = min(target_depth, current_depth + rop / 60)  # 1-minute readings
            current_time += timedelta(minutes=1)
    
    return pd.DataFrame(drilling)

df_drilling = generate_drilling_parameters(num_wells=3, start_time=START_DATE, hours=2)  # 2 hours for demo
print(f"Generated {len(df_drilling)} drilling parameter records")
print(f"Kick indicators: {df_drilling['kick_indicator'].sum()}")
df_drilling.head()

## 6. Generate Production Events

In [ ]:
def generate_production_events(wells_df, days=30):
    """Generate operational events"""
    events = []
    
    event_types = [
        ('Shutdown', 0.15),
        ('Startup', 0.15),
        ('Well Test', 0.20),
        ('Workover', 0.10),
        ('ESP Failure', 0.08),
        ('Rod Pump Failure', 0.07),
        ('Choke Change', 0.10),
        ('Chemical Treatment', 0.08),
        ('Safety Alarm', 0.05),
        ('Tank Truck Dispatch', 0.02)
    ]
    
    for _, well in wells_df.iterrows():
        # Random number of events per well
        num_events = random.randint(0, 5)
        
        for _ in range(num_events):
            event_type = random.choices([e[0] for e in event_types], 
                                       weights=[e[1] for e in event_types])[0]
            event_time = fake.date_time_between(start_date='-30d', end_date='now')
            
            # Duration depends on event type
            duration_hours = {
                'Shutdown': random.randint(1, 72),
                'Startup': 1,
                'Well Test': random.randint(4, 24),
                'Workover': random.randint(24, 336),  # 1-14 days
                'ESP Failure': random.randint(48, 720),
                'Rod Pump Failure': random.randint(24, 168),
                'Choke Change': 1,
                'Chemical Treatment': random.randint(2, 8),
                'Safety Alarm': random.randint(1, 4),
                'Tank Truck Dispatch': 2
            }.get(event_type, 4)
            
            # Production impact
            production_loss = {
                'Shutdown': well['current_oil_bopd'] * duration_hours / 24,
                'Workover': well['current_oil_bopd'] * duration_hours / 24,
                'ESP Failure': well['current_oil_bopd'] * duration_hours / 24,
                'Rod Pump Failure': well['current_oil_bopd'] * duration_hours / 24
            }.get(event_type, 0)
            
            events.append({
                'event_id': str(uuid.uuid4()),
                'well_id': well['well_id'],
                'event_type': event_type,
                'start_time': event_time,
                'end_time': event_time + timedelta(hours=duration_hours),
                'duration_hours': duration_hours,
                'production_loss_bbl': round(production_loss, 0),
                'cost_usd': random.randint(1000, 500000) if event_type in ['Workover', 'ESP Failure', 'Rod Pump Failure'] else random.randint(100, 5000),
                'root_cause': fake.sentence() if event_type in ['ESP Failure', 'Rod Pump Failure', 'Shutdown'] else None,
                'operator': fake.name(),
                'notes': fake.sentence()
            })
    
    return pd.DataFrame(events)

df_events = generate_production_events(df_wells)
print(f"Generated {len(df_events)} production events")
print(f"Event types: {df_events['event_type'].value_counts().to_dict()}")
print(f"Total production loss: {df_events['production_loss_bbl'].sum():,.0f} barrels")
df_events.head()

## 7. Data Validation

In [ ]:
print("=== Data Validation Report ===")
print(f"\nWells: {len(df_wells)} records")
print(f"  - By status: {df_wells['status'].value_counts().to_dict()}")
print(f"  - By lift type: {df_wells['lift_type'].value_counts().to_dict()}")
print(f"  - Total production: {df_wells['current_oil_bopd'].sum():,.0f} BOPD")

print(f"\nEquipment: {len(df_equipment)} records")
print(f"  - By type: {df_equipment['equipment_type'].value_counts().to_dict()}")
print(f"  - Valid FK: {df_equipment['well_id'].isin(df_wells['well_id']).all()}")

print(f"\nProduction Telemetry: {len(df_production)} records")
print(f"  - Anomalies: {df_production['has_anomaly'].sum()}")

print(f"\nESP Telemetry: {len(df_esp)} records")
print(f"  - Failure risks: {df_esp['failure_risk_flag'].sum()}")

print(f"\nRod Pump Cards: {len(df_cards)} records")
print(f"  - Conditions: {df_cards['diagnosed_condition'].value_counts().to_dict()}")

print(f"\nDrilling Parameters: {len(df_drilling)} records")
print(f"  - Kick indicators: {df_drilling['kick_indicator'].sum()}")

print(f"\nProduction Events: {len(df_events)} records")
print(f"  - Total prod loss: {df_events['production_loss_bbl'].sum():,.0f} barrels")
print(f"  - Total cost: ${df_events['cost_usd'].sum():,.0f}")

## 8. Write to Lakehouse

In [ ]:
# Uncomment for Fabric execution
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.getOrCreate()

# # Note: position_data and load_data columns contain arrays - may need special handling
# df_cards_write = df_cards.drop(columns=['position_data', 'load_data'])  # Remove array columns for simple write

# spark.createDataFrame(df_wells).write.mode("overwrite").format("delta").saveAsTable("dim_wells")
# spark.createDataFrame(df_equipment).write.mode("overwrite").format("delta").saveAsTable("dim_equipment")
# spark.createDataFrame(df_production).write.mode("append").format("delta").saveAsTable("fact_production_telemetry")
# spark.createDataFrame(df_esp).write.mode("append").format("delta").saveAsTable("fact_esp_telemetry")
# spark.createDataFrame(df_cards_write).write.mode("append").format("delta").saveAsTable("fact_rod_pump_cards")
# spark.createDataFrame(df_drilling).write.mode("append").format("delta").saveAsTable("fact_drilling_parameters")
# spark.createDataFrame(df_events).write.mode("append").format("delta").saveAsTable("fact_production_events")

# Local testing
df_wells.to_csv('dim_wells.csv', index=False)
df_production.to_csv('fact_production_telemetry.csv', index=False)
print("Data saved successfully")

## Known Limitations
1. Dynamometer cards are simplified - real cards have complex fluid dynamics
2. Decline curves are simplified exponential - actual decline varies by basin
3. Kick detection is random - real kicks have precursor signatures
4. No correlation between events and sensor anomalies

## Next Steps
1. Build ESP run life prediction model
2. Train rod pump card classification model
3. Implement real-time kick detection algorithm
4. Connect to Digital Twin Builder for field modeling